# 6.2 Steps and Type 2 Diabetes

**Wearables series, class 2.** Do people who take more steps a day go on to develop type 2 diabetes **less often**? In this class we follow every *All of Us* participant with Fitbit data **forward in time**: we measure their daily steps first, then watch who is diagnosed with type 2 diabetes later, and when. It's the design of a published *All of Us* study, and it fixes the biggest problem we met in **5.2**: answers given **after** a diagnosis.

### By the end of this notebook you'll be able to

1. Explain a **prospective cohort** design, and why it avoids reverse causation
2. Build **baseline** steps, eligibility, and **follow-up** from dates, mostly inside BigQuery
3. Calculate **incidence rates** per 1,000 person-years
4. Draw **Kaplan–Meier** curves and fit a **Cox model** for a new outcome (from **3.4**)
5. Choose what to adjust for: **confounders** (age, sex) vs a possible **mediator** (BMI)
6. Test the result with a **sensitivity analysis** (a lag period)

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Among adults **without** diabetes, do those who take more steps a day develop type 2 diabetes less often? |
| 🧪 **The analysis** | Baseline steps over each person's first 6 months of Fitbit data; follow-up for a **new** type 2 diabetes diagnosis; incidence rates; Kaplan–Meier curves; **Cox** models adjusted for age and sex, then BMI; a 1-year lag |
| 🎯 **Why** | Steps measured **before** any diagnosis can't be the result of the diagnosis. Survival methods use everyone's follow-up, however long |
| ✅ **What we conclude** | How the rate of new type 2 diabetes differs by daily steps, as hazard ratios |
| ⚠️ **What it can't tell us** | Cause. People who walk more differ in many ways (income, health, diet). And Fitbit users aren't typical |
| 🔗 **Connections** | Valid days and costs from **6.1**; the cohort query from **1.1**; survival methods from **3.4**; reverse causation from **5.2**; confounders and mediators from **5.3** |

| | |
|---|---|
| 📂 **Uses** | the Controlled Tier dataset: the shared cohort query, Fitbit daily tables, BMI measurements |
| 💾 **Saves** | `steps_followup.csv` (one row per person in the analysis) |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts: the design | 20 |
| 2. Setup and the cohort | 5 |
| 3. Baseline steps, in BigQuery | 15 |
| 4. Who's in the analysis, and follow-up | 15 |
| 5. Incidence rates and Kaplan–Meier curves | 15 |
| 6. Cox models: what to adjust for | 15 |
| 7. A sensitivity analysis | 10 |
| Wrap-up | 5 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 95 minutes |
| ⏱️ **Run time** | about **5 minutes** to run every cell |
| 💰 **Data cost** | about **$1 to $2**: the cohort query, and one query that builds everyone's baseline steps and BMI inside BigQuery. Both are priced before they run |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| **Estimated total** | about **$2** per student for this class |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### Why not use our sample of cases and controls?

Our 5,000 cases were chosen **because** they already had type 2 diabetes. For most, Fitbit data starts **after** their diagnosis, so their steps describe life **with** diabetes: perhaps walking more on a doctor's advice, or less because of complications. That's **reverse causation** (**5.2**). And only a few hundred cases have Fitbit data at all (**6.1**).

### A prospective cohort: measure first, then follow

Instead, we start with **everyone** in *All of Us* who has Fitbit data, and **follow them forward**:

```
first valid           end of baseline                          end of follow-up
Fitbit day            (6 months later)                         (last EHR visit, death, or 2025)
    |──── baseline ────|──────────────── follow-up ─────────────────────|
      median daily          does type 2 diabetes appear? when?
      steps
excluded: anyone with type 2 diabetes (or another kind of diabetes) before the end of baseline
```

**This design avoids reverse causation by construction:** steps are measured **before** any diagnosis, so the diagnosis can't have changed them. It's the design of the published *All of Us* study of step counts and chronic disease (Master et al., *Nature Medicine*, 2022).

| Term | Means |
|---|---|
| **Baseline** | the period when we measure the exposure: here, each person's first 6 months of valid Fitbit days |
| **Exposure** | what might affect the outcome: **median daily steps** at baseline |
| **Incident** case | a **new** diagnosis, during follow-up (a **prevalent** case already had it at baseline: excluded) |
| **Follow-up** | from the end of baseline until diagnosis, or **censoring**: the last EHR visit, death, or the end of the data |
| **Person-years** | the total time everyone was followed: 100 people for 2 years = 200 person-years |
| **Incidence rate** | new cases ÷ person-years, often per **1,000 person-years** |
| **Hazard ratio** (**3.4**) | the rate of new cases in one group ÷ another, at any moment: HR 0.7 = 30% lower |

### What to adjust for

| Variable | Role | Adjust? |
|---|---|---|
| **Age**, **sex** | affect both steps and diabetes risk: **confounders** | yes |
| **BMI** | a confounder (heavier people may walk less) **and** a possible **mediator** (walking → lower weight → less diabetes) | show **both** models, and explain the difference |

If BMI is partly a **step on the path** (**5.3**), adjusting for it removes part of the effect of walking. So we report the model **without** BMI as the main result, and **with** BMI as "the part of the link not explained by weight".

### Still not proof

People who walk more also tend to have higher incomes, fewer illnesses, and healthier diets. Some will have been less active because of **undiagnosed** illness, including early diabetes. Part 7 tests that last one with a **lag**.

## Part 2. Setup and the cohort

We use the shared cohort query (`sql/t2d_cohort.sql`, **1.1**) for **everyone**, not our sample: it gives each person's case or control status, the date of their first type 2 diabetes diagnosis, and their EHR dates.

💰 Cents to under $1.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions
library(survival)          # survival analysis (3.4)
library(broom)             # tidy model results

cohort <- run_query(str_glue(read_file("../sql/t2d_cohort.sql"), AGE_DATE = "2025-01-01")) %>%
  mutate(person_id = as.character(person_id))

cohort %>% count(cohort)

**🔍 Read the output:** cases, controls, and excluded people (type 1 diabetes, or unclear), as in **1.1**: over 400,000 adults with EHR data.

## Part 3. Baseline steps, in BigQuery

For about 68,000 Fitbit users, downloading every day (tens of millions of rows) would be slow. Instead, **BigQuery** does the work, and returns **one row per person**. The query is a chain of named steps (**1.1**):

| Step | Does |
|---|---|
| `wear` | wear minutes per person per day (the heart-rate zone total, **6.1**) |
| `valid` | valid days: 10+ hours of wear, 100 to 45,000 steps (**6.1**) |
| `first_day` | each person's first valid day |
| `baseline` | the **6 months** from that day: number of valid days, and **median** daily steps |
| `bmi` | each person's most recent BMI in the 2 years before the end of baseline |
| final `SELECT` | people with at least **10** valid baseline days |

`APPROX_QUANTILES(steps, 100)[OFFSET(50)]` is BigQuery's fast median: the 50th of 100 percentiles. `ARRAY_AGG(… ORDER BY date DESC LIMIT 1)[OFFSET(0)]` keeps the **latest** value.

In [ ]:
baseline_sql <- str_glue("
WITH wear AS (
  SELECT person_id, date, SUM(minute_in_zone) AS wear_minutes
  FROM `{CURRENT_DATASET}.heart_rate_summary`
  GROUP BY person_id, date
),
valid AS (
  SELECT a.person_id, a.date, a.steps
  FROM `{CURRENT_DATASET}.activity_summary` AS a
  JOIN wear AS w ON a.person_id = w.person_id AND a.date = w.date
  WHERE w.wear_minutes >= 600 AND a.steps BETWEEN 100 AND 45000
),
first_day AS (
  SELECT person_id, MIN(date) AS first_valid FROM valid GROUP BY person_id
),
baseline AS (
  SELECT v.person_id,
         MIN(v.date) AS baseline_start,
         MAX(v.date) AS baseline_end,
         COUNT(*)    AS baseline_days,
         APPROX_QUANTILES(v.steps, 100)[OFFSET(50)] AS baseline_steps
  FROM valid AS v
  JOIN first_day AS f ON v.person_id = f.person_id
  WHERE v.date < DATE_ADD(f.first_valid, INTERVAL 6 MONTH)
  GROUP BY v.person_id
),
bmi AS (
  SELECT b.person_id,
         ARRAY_AGG(m.value_as_number ORDER BY m.measurement_date DESC LIMIT 1)[OFFSET(0)] AS bmi
  FROM baseline AS b
  JOIN `{CURRENT_DATASET}.measurement` AS m ON m.person_id = b.person_id
  WHERE m.measurement_concept_id = 3038553                      -- BMI
    AND m.value_as_number BETWEEN 12 AND 80                     -- plausible values (2.1)
    AND m.measurement_date BETWEEN DATE_SUB(b.baseline_end, INTERVAL 2 YEAR) AND b.baseline_end
  GROUP BY b.person_id
)
SELECT b.*, bmi.bmi
FROM baseline AS b
LEFT JOIN bmi ON b.person_id = bmi.person_id
WHERE b.baseline_days >= 10")

baseline <- run_if_cheap(baseline_sql, max_dollars = 3) %>%
  mutate(person_id = as.character(person_id))

last_query_cost()

baseline %>%
  summarise(people = n(),
            median_baseline_days = median(baseline_days),
            median_steps = median(baseline_steps),
            pct_with_bmi = round(100 * mean(!is.na(bmi)), 1),
            first_baseline = min(baseline_start), last_baseline = max(baseline_end))

**🔍 Read the output:** the cost, then how many Fitbit users have a usable baseline (10+ valid days in their first 6 months), how many valid days they typically have, their median daily steps, how many have a recent BMI, and the range of dates.

Most of the cost is the `measurement` table (for BMI), not the Fitbit tables: BigQuery reads the `measurement` columns for everyone, even though we keep only BMI.

## Part 4. Who's in the analysis, and follow-up

Now we join baseline steps to the cohort, and apply the rules, **counting** how many people each rule removes (as in **2.1**):

In [ ]:
end_of_data <- as.Date("2025-01-01")    # the CDR v9 data cutoff

joined <- baseline %>%
  inner_join(cohort, by = "person_id") %>%
  mutate(age_at_baseline = as.numeric(baseline_end - as.Date(date_of_birth)) / 365.25,
         followup_end    = pmin(as.Date(last_ehr_date), as.Date(death_date), end_of_data, na.rm = TRUE),
         t2d_date        = as.Date(first_t2d_date))

steps_flow <- tibble(
  rule = c("Fitbit baseline, and in the cohort query (adults with EHR)",
           "not 'excluded' (type 1 or unclear diabetes)",
           "no type 2 diabetes by the end of baseline",
           "18 or older at the end of baseline",
           "some EHR follow-up after baseline"),
  people = c(
    nrow(joined),
    sum(joined$cohort != "excluded"),
    sum(joined$cohort != "excluded" & !(joined$cohort == "case" & joined$t2d_date <= joined$baseline_end)),
    sum(joined$cohort != "excluded" & !(joined$cohort == "case" & joined$t2d_date <= joined$baseline_end) &
        joined$age_at_baseline >= 18),
    NA))

followup <- joined %>%
  filter(cohort != "excluded",
         !(cohort == "case" & t2d_date <= baseline_end),        # prevalent cases out
         age_at_baseline >= 18,
         followup_end > baseline_end) %>%                        # needs follow-up time
  mutate(event     = as.integer(cohort == "case" & t2d_date <= followup_end),
         exit_date = if_else(event == 1, t2d_date, followup_end),
         years     = as.numeric(exit_date - baseline_end) / 365.25)

steps_flow$people[5] <- nrow(followup)
steps_flow

**🔍 Read the output:** a **flow table**: how many people are left after each rule. Papers report this so readers know exactly who was studied.

- Some Fitbit users have no EHR data, so they aren't in the cohort query at all.
- The biggest drop is usually **prevalent** type 2 diabetes: people already diagnosed before or during baseline.
- The last rule removes people whose health records end before their baseline does: no follow-up to watch.

How long were people followed, and how many developed type 2 diabetes?

In [ ]:
followup %>%
  summarise(people = n(),
            new_cases = sum(event),
            person_years = round(sum(years)),
            median_followup_years = round(median(years), 1),
            rate_per_1000_py = round(1000 * sum(event) / sum(years), 1))

**🔍 Read the output:** the number of people, **new** type 2 diabetes diagnoses during follow-up, total person-years, median follow-up, and the overall **incidence rate** per 1,000 person-years.

A median follow-up of a few years is typical: many people started sharing Fitbit data only recently.

## Part 5. Incidence rates and Kaplan–Meier curves

### Step groups

We put people into four groups by baseline steps, in round numbers that are easy to communicate. The lowest group is the **reference**.

In [ ]:
followup <- followup %>%
  mutate(steps_group = cut(baseline_steps,
                           breaks = c(0, 5000, 7500, 10000, Inf), right = FALSE,
                           labels = c("under 5,000", "5,000 to 7,499", "7,500 to 9,999", "10,000 or more")),
         steps_1000  = baseline_steps / 1000,
         age10       = age_at_baseline / 10)

rates <- followup %>%
  group_by(steps_group) %>%
  summarise(people = n(), n = sum(event), person_years = sum(years)) %>%
  mutate(rate_per_1000_py = round(1000 * n / person_years, 1)) %>%
  hide_small_counts() %>%                                              # hide new cases of 1 to 20
  mutate(rate_per_1000_py = if_else(is.na(n), NA_real_, rate_per_1000_py),   # and their rates
         person_years = round(person_years)) %>%
  rename(new_cases = n)

rates

**🔍 Read the output:** for each step group: people, new cases, person-years, and the **incidence rate** per 1,000 person-years. Look for a **gradient**: does the rate fall as steps rise?

These rates are **unadjusted**: the groups also differ in age and sex. Rates hide counts of 1 to 20 (and the rate, which would reveal them).

### Kaplan–Meier curves

The share of people **still without** type 2 diabetes over follow-up, by step group (as in **3.4**). The curves are cut where fewer than 50 people in any group are still being followed: beyond that, they're unreliable, and could describe very few people.

In [ ]:
km_fit <- survfit(Surv(years, event) ~ steps_group, data = followup)

km_df <- tidy(km_fit) %>%
  mutate(group = str_remove(strata, "steps_group=")) %>%
  bind_rows(tibble(time = 0, estimate = 1, conf.low = 1, conf.high = 1, n.risk = Inf,
                   group = levels(followup$steps_group))) %>%
  mutate(group = factor(group, levels = levels(followup$steps_group)))

max_years <- km_df %>%                                    # where every group still has 50+ at risk
  group_by(group) %>%
  summarise(t = max(time[n.risk >= 50])) %>%
  pull(t) %>%
  min()

ggplot(km_df %>% filter(time <= max_years), aes(time, estimate, colour = group)) +
  geom_step(linewidth = 0.9) +
  scale_colour_brewer(palette = "Dark2") +
  theme_bw(base_size = 13) +
  labs(x = "Years after baseline", y = "Share still without type 2 diabetes", colour = "Daily steps",
       title = "Kaplan-Meier: new type 2 diabetes, by baseline daily steps")

survdiff(Surv(years, event) ~ steps_group, data = followup)

**🔍 Read the output:** each curve starts at 100% and steps **down** at each new diagnosis. Lower curves = more new diabetes. The **log-rank test** (`survdiff`) asks whether the curves differ.

The y-axis is zoomed in: the drops are small, because only a small share of people develop diabetes in a few years.

### Number at risk

In [ ]:
summary(km_fit, times = 0:floor(max_years), extend = TRUE) %>%
  with(tibble(group = str_remove(strata, "steps_group="), years = time, n = n.risk)) %>%
  hide_small_counts() %>%
  pivot_wider(names_from = years, values_from = n, names_prefix = "year_")

**🔍 Read the output:** how many people in each group were still being followed at each year. The numbers fall quickly: most people's follow-up is short.

## Part 6. Cox models: what to adjust for

Three models, on the **same people** (those with a BMI, so the comparison is fair, as in **5.3**):

1. **Age and sex**: the main result
2. **+ BMI**: the part of the link **not** explained by weight
3. **Steps per 1,000**, continuous: the change in rate for every extra 1,000 steps a day (age and sex adjusted)

In [ ]:
cox_data <- followup %>%
  filter(sex_at_birth %in% c("Female", "Male"), !is.na(bmi)) %>%
  mutate(sex_at_birth = factor(sex_at_birth), bmi5 = bmi / 5)

c(people = nrow(cox_data), new_cases = sum(cox_data$event))

m_age_sex <- coxph(Surv(years, event) ~ steps_group + age10 + sex_at_birth, data = cox_data)
m_bmi     <- coxph(Surv(years, event) ~ steps_group + age10 + sex_at_birth + bmi5, data = cox_data)
m_per1000 <- coxph(Surv(years, event) ~ steps_1000 + age10 + sex_at_birth, data = cox_data)

bind_rows(
  tidy(m_age_sex, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "1. age and sex"),
  tidy(m_bmi,     exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "2. + BMI"),
  tidy(m_per1000, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "3. per 1,000 steps")) %>%
  filter(str_detect(term, "steps")) %>%
  transmute(model, term = str_remove(term, "steps_group"),
            HR = sprintf("%.2f (%.2f to %.2f)", estimate, conf.low, conf.high),
            p_value = format.pval(p.value, digits = 2, eps = 0.001))

**🔍 Read the output:** hazard ratios for type 2 diabetes, compared with people taking **under 5,000** steps a day (models 1 and 2), or per extra 1,000 steps (model 3).

- **Model 1:** HRs **below 1** mean a lower rate of new diabetes than the least active group. Look for a **gradient**: lower HRs with more steps.
- **Model 2:** if the HRs move **toward 1** after adding BMI, part of the link runs through (or is confounded by) weight.
- **Model 3:** one number for the whole range: e.g. HR 0.90 per 1,000 steps = a 10% lower rate for every extra 1,000 steps a day.

### Check the proportional hazards assumption (3.4)

In [ ]:
cox.zph(m_age_sex)

**🔍 Read the output:** a small p-value for a variable means its hazard ratio may **change over time**. With large samples, small p-values are common for modest changes; look at the size, and (for a paper) plot it with `plot(cox.zph(m_age_sex))`.

### ✍️ Your turn

Compare people **in** the analysis with **all** adults in the cohort query: age at the data cutoff, percent female, percent cases. Who uses a Fitbit, and who might our result not apply to?

<details>
<summary><b>Show answer</b></summary>

```r
cohort %>%
  filter(cohort != "excluded") %>%
  mutate(in_analysis = person_id %in% followup$person_id) %>%
  group_by(in_analysis) %>%
  summarise(people = n(), mean_age = round(mean(age), 1),
            pct_female = round(100 * mean(sex_at_birth == "Female", na.rm = TRUE), 1),
            pct_case = round(100 * mean(cohort == "case"), 1))
```

Fitbit users are a **selected** group (**6.1**): the result describes them. A link this strong in Fitbit users may be weaker, stronger, or different in people who don't use wearables.

</details>

In [ ]:
# Your code here

## Part 7. A sensitivity analysis: a 1-year lag

Someone with **undiagnosed** diabetes, or another illness, might already walk less at baseline, and be diagnosed soon after. That's reverse causation sneaking back in. A common check: **ignore the first year** of follow-up. People diagnosed in that year are dropped, and everyone's follow-up starts 1 year after baseline.

In [ ]:
lagged <- cox_data %>%
  filter(years > 1) %>%
  mutate(years = years - 1)

c(people = nrow(lagged), new_cases = sum(lagged$event))

m_lag <- coxph(Surv(years, event) ~ steps_group + age10 + sex_at_birth, data = lagged)

bind_rows(
  tidy(m_age_sex, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "main"),
  tidy(m_lag,     exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "1-year lag")) %>%
  filter(str_detect(term, "steps")) %>%
  transmute(term = str_remove(term, "steps_group"), model,
            HR = sprintf("%.2f (%.2f to %.2f)", estimate, conf.low, conf.high)) %>%
  pivot_wider(names_from = model, values_from = HR)

save_data(followup %>% select(person_id, cohort, baseline_start, baseline_end, baseline_days,
                              baseline_steps, steps_group, bmi, age_at_baseline, sex_at_birth,
                              event, years),
          "steps_followup.csv")

**🔍 Read the output:** the hazard ratios with and without the first year. If they barely change, early undiagnosed illness doesn't explain the link. If they move toward 1, part of it did.

`steps_followup.csv` saves the analysis data for **6.3**, which uses the same design for sleep and heart rate.

**💬 Discuss:**

1. Steps were measured over **6 months**, but follow-up lasts years. People's activity changes. How might that affect the result?
2. Type 2 diabetes is diagnosed in health care. Could people who walk more also see doctors **less** (and be diagnosed later)? How would that bias the result?
3. What would make you more confident that walking itself lowers diabetes risk? (Other studies, a dose–response gradient, a randomised trial, a plausible mechanism.)

## ⚠️ Common errors

| Message or problem | Means | Fix |
|---|---|---|
| *Not run: over the limit* | the baseline query costs more than `max_dollars` | check the printed estimate; raise the limit only knowingly |
| `max_years` is `-Inf` or a warning about `max()` | a step group never had 50 people at risk | check `rates`: the group may be too small; combine groups |
| A step group has `NA` rates | it had 1 to 20 new cases, hidden | combine groups, or report that it was too small |
| *Loglik converged before variable …* in `coxph()` | a group with very few events | combine small groups |
| Negative or zero `years` | follow-up ended before baseline did | the `followup_end > baseline_end` filter removes these: check it ran |

## ✅ Key takeaways

- A **prospective** design measures the exposure **before** the outcome: it avoids reverse causation by construction
- Build baseline, eligibility, and follow-up from **dates**, and report a **flow table** of who was excluded and why
- **Incidence rates** per 1,000 person-years, **Kaplan–Meier** curves, and **Cox** models answer "how often" and "how soon"
- Adjust for **confounders**; show models with and without a possible **mediator** (BMI)
- A **lag** tests whether undiagnosed illness explains the result
- It's still an **association**, in a **selected** group

**Next:** **6.3 sleep_and_heart_rate**: the same design for sleep and resting heart rate.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).

## 🧑‍🏫 Instructor notes

- **Draw the timeline** before anything else: baseline, then follow-up, with prevalent cases excluded. Most of the class follows from it.
- **"Avoids reverse causation by construction"** is the key phrase: compare with 5.2, where most answers came after diagnosis.
- **BMI: confounder or mediator?** Ask students to argue both sides before showing model 2.
- **Kaplan–Meier curves end early** on purpose: where few people are left, curves describe very few people and jump around.
- **Likely question:** "Is 10,000 steps a day magic?" No: the number came from a 1960s Japanese pedometer advertisement. Look at the **gradient** across groups instead.